# One-day setups: pre-registration (Phase 13, step 2)

This notebook contains **only a pre-registration**: two new long-only setups for one-day holds on BTC/USDT, the development-period thresholds they use, development-period frequency counts, and the evaluation plan.

**No outcome has been computed.** No return of any trade or day after a signal, no outcome statistic, and no relationship between any condition and price direction appears here or was computed before writing it. The numbers computed are:
- one quantile of a signal-day volatility measure over the 2017–2023 development period;
- counts of how many development days each setup would trade.

The existing continuation and pullback setups are not reused.

**What the reasons may draw on.** Notebooks 10–14 measured volatility and counts only:
- **The cost is small against a one-day move:** 0.30% round trip was 0.085 of a typical one-day move overall (typical move 3.52%), and 0.109–0.138 in 2023–2026 (notebook 14).
- **Saturday entries are the calmest days:**
  - in 2017–2023 the typical Saturday move was 2.72%, the lowest of any weekday (cost ratio 0.110, against 0.059–0.098 on other days);
  - in 2024 onward it was 1.23% (cost ratio 0.244, against 0.106–0.147).
- **Active US opens are followed by larger moves.** Days after a highly active US open (09:00–10:59 New York) had larger one-day moves: in 2017–2023 the high-activity tercile had a typical move of 4.68% (cost ratio 0.064) against 2.78% for the low tercile (0.108); 2024 onward, 2.97% (0.101) against 2.24% (0.134).
- **Intraday timing:** 1-hour moves are largest around the US open (notebooks 11–12).

No one-day price-direction result exists, so the reasons below rest on these facts and on general market reasoning.

## Common rules (both setups)

**Trades.**
- A signal at the daily close of day t (UTC) buys at the open of day t + 1 and sells at the open of day t + 2.
- Trades come from `find_trades(signal, 1)` in `cryptosense.backtest`, unchanged: one position at a time. A signal on day t + 1, while the trade from day t is still open, is ignored, even though that trade exits at the open where the next one would enter.
- Long only, all-in, no leverage.

**Costs.** 0.30% round trip: 0.15% per side (0.10% fee + 0.05% slippage). The net log return of a trade is log(exit × (1 − 0.0015) / (entry × (1 + 0.0015))), the same cost convention as `run_backtest`.

**Signal-day US-open measures** (from 15-minute candles, on the UTC day t):
- The window is the 8 15-minute candles opening from 09:00 to 10:45 New York time (13:00–14:45 UTC in US summer time, 14:00–15:45 UTC otherwise). It ends at 11:00 New York, before the daily close of day t, so it is known at the signal.
- Each candle's 15-minute log return is its close against the previous candle's close, as in `add_intraday_features`.
- `rv_open` = square root of the sum of the 8 squared returns: how active the US open was.
- `ret_open` = sum of the 8 returns = log(close at 11:00 New York / close at 09:00 New York): the net price change over the US open.
- If any of the 8 returns is missing (a missing candle, including the 08:45 candle the first return needs), both measures are NaN and day t cannot signal.

**Daily inputs.** `add_regimes(add_features(...))` on daily data: `trend` and `log_return` (close of day t against close of day t − 1). A day with a missing input never signals.

**Saturday entries are excluded in both setups** (signals at the close of a Friday, UTC, are not traded).
- In the development period, Saturday was the calmest entry day (typical move 2.72%, against 3.06–5.09% on other days), so a Saturday trade has the least room to cover the 0.30% cost.
- Both setups rest on activity from the US session, which is closed at the weekend, so the mechanism they assume is weakest on Saturday.
- Sunday (the second-calmest day) is not excluded. Only the clear outlier is removed, to avoid stacking filters.
- 2024 onward showed the same ordering (Saturday 1.23%). The exclusion is justified by the development figures alone.

**Threshold.** Both setups use one quantile-based threshold: the 2/3 quantile of `rv_open` over development signal days.
- Development signal days are days t whose trade would exit (day t + 2) on or before 2023-12-31, with a complete window.
- All weekdays are included in the quantile, so it matches the terciles of notebook 14.
- "High activity" means the top third: a conventional split, and the one used in notebook 14's exploration. No other quantile was tried.

The threshold is computed and printed below.

## Development-period threshold

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from cryptosense.data import load_processed
from cryptosense.features import add_features
from cryptosense.regimes import add_regimes
from cryptosense.intraday import NY_TZ, lag_by_time
from cryptosense.backtest import find_trades

DATA_DIR = Path.cwd().parent / "data"
SYMBOL = "BTC/USDT"
DEV_LAST_EXIT = pd.Timestamp("2023-12-31", tz="UTC")
WINDOW_HOURS_NY = [9, 10]
CANDLES_IN_WINDOW = 8
EXCLUDED_ENTRY_WEEKDAY = "Saturday"

daily = add_regimes(add_features(load_processed(SYMBOL, DATA_DIR)))
m15 = load_processed(SYMBOL, DATA_DIR, timeframe="15m")

# Signal-day US-open measures from the 8 window returns (NaN unless all 8 exist).
log_close = np.log(m15["close"])
ret_15m = log_close - lag_by_time(log_close, "15min")
in_window = np.isin(m15.index.tz_convert(NY_TZ).hour, WINDOW_HOURS_NY)
w = pd.DataFrame({"r": ret_15m[in_window], "day": m15.index[in_window].normalize()})
per_day = w.groupby("day")["r"].agg(
    rv_open=lambda r: np.sqrt((r ** 2).sum()), ret_open="sum", returns="count"
)
complete = per_day["returns"] == CANDLES_IN_WINDOW
per_day.loc[~complete, ["rv_open", "ret_open"]] = np.nan
per_day.index = per_day.index.as_unit(daily.index.unit)
daily = daily.join(per_day[["rv_open", "ret_open"]])

# Exit day (t + 2) and entry weekday (t + 1) for every possible signal day t.
daily["exit_day"] = pd.Series(daily.index, index=daily.index).shift(-2)
daily["entry_weekday"] = pd.Series(daily.index + pd.Timedelta(days=1), index=daily.index).dt.day_name()

dev_signal_days = daily[daily["exit_day"] <= DEV_LAST_EXIT]
dev_rv = dev_signal_days["rv_open"].dropna()
RV_OPEN_HIGH = dev_rv.quantile(2 / 3)

print(f"development signal days: {dev_signal_days.index[0].date()} -> {dev_signal_days.index[-1].date()} "
      f"({len(dev_signal_days):,} days; {len(dev_rv):,} with a complete US-open window)")
print(f"rv_open quantiles (development): 1/3 = {dev_rv.quantile(1 / 3):.6f}, 2/3 = {RV_OPEN_HIGH:.6f}")
print(f"\nThreshold used by both setups: rv_open > {RV_OPEN_HIGH:.6f}")

development signal days: 2017-08-17 -> 2023-12-29 (2,326 days; 2,320 with a complete US-open window)
rv_open quantiles (development): 1/3 = 0.005761, 2/3 = 0.010688

Threshold used by both setups: rv_open > 0.010688


## Setup D1 — Strong, active US open in an uptrend

**Idea.** In a daily uptrend, when the US open on day t was highly active and price rose over it, buy at the next day's open and hold for one day.

**Why it might work.**
- **Room to cover costs:** days after a highly active US open had the largest one-day moves in notebook 14, so a trade there has the most room to cover the 0.30% cost.
- **Persistent flows (general market reasoning):** a rise during the most active part of the trading day reflects buying by US participants, such as funds and ETF flows. Such flows are often spread over more than one session, so they may continue into the next day.
- **Trend filter:** restricting to days in a daily uptrend keeps long trades on the side of the prevailing trend.

This is a hypothesis; no one-day price-direction result supports or contradicts it.

**Conditions** (all on signal day t):
1. `trend == "up"`
2. `rv_open > RV_OPEN_HIGH`: top third of development US-open activity.
3. `ret_open > 0`: price rose from 09:00 to 11:00 New York.
4. The entry day (t + 1) is not a Saturday.

**Threshold justification.** `ret_open > 0` is a sign condition. `RV_OPEN_HIGH` is the development 2/3 quantile printed above. `trend` is the existing daily regime label.

## Setup D2 — Active US-open sell-off recovered by the close

**Idea.** When the US open on day t was highly active and price fell over it, but day t still closed above the previous day's close, buy at the next day's open and hold for one day.

**Why it might work.**
- **Room to cover costs:** as for D1, a highly active US open marks days followed by larger one-day moves.
- **Selling absorbed (general market reasoning):** a sharp, active sell-off at the US open that is fully recovered by the close suggests the selling was absorbed by buyers during the rest of the session. A day that closes up after an early sell-off is often read as a sign of underlying demand.
- **No trend filter:** the condition describes the day itself, so D2 is distinct from the existing pullback setup.

This is a hypothesis; no one-day price-direction result supports or contradicts it.

**Conditions** (all on signal day t):
1. `rv_open > RV_OPEN_HIGH`: top third of development US-open activity.
2. `ret_open < 0`: price fell from 09:00 to 11:00 New York.
3. `log_return > 0`: day t closed above the close of day t − 1.
4. The entry day (t + 1) is not a Saturday.

**Threshold justification.** `ret_open < 0` and `log_return > 0` are sign conditions. `RV_OPEN_HIGH` is the same development 2/3 quantile.

**Overlap.** D1 requires `ret_open > 0` and D2 requires `ret_open < 0`, so they never signal on the same day.

## Development-period frequency counts (counts only, no returns)

For each setup:
- **Signal days:** development days on which every condition holds.
- **Trades:** the trades `find_trades(signal, 1)` makes from those signal days.
- **Skipped for missing data:** non-Saturday-entry days whose window is incomplete or that have a missing input.

Development = trades exiting on or before 2023-12-31. No price after a signal is used.

In [2]:
SETUPS_D = {
    "D1": {"inputs": ["trend", "rv_open", "ret_open"],
           "rule": lambda d: (d["trend"] == "up") & (d["rv_open"] > RV_OPEN_HIGH) & (d["ret_open"] > 0)},
    "D2": {"inputs": ["rv_open", "ret_open", "log_return"],
           "rule": lambda d: (d["rv_open"] > RV_OPEN_HIGH) & (d["ret_open"] < 0) & (d["log_return"] > 0)},
}

not_saturday = daily["entry_weekday"] != EXCLUDED_ENTRY_WEEKDAY
rows = []
for name, setup in SETUPS_D.items():
    usable = daily[setup["inputs"]].notna().all(axis=1)
    signal = (usable & setup["rule"](daily).fillna(False).astype(bool) & not_saturday)
    trades = find_trades(signal, 1)
    trade_signal_days = daily.index[[entry - 1 for entry, _ in trades]]
    dev_mask = daily["exit_day"] <= DEV_LAST_EXIT
    dev_trades = [d for d in trade_signal_days if daily.loc[d, "exit_day"] <= DEV_LAST_EXIT]
    rows.append({
        "setup": name,
        "development days": int(dev_mask.sum()),
        "excluded (Saturday entry)": int((dev_mask & ~not_saturday).sum()),
        "skipped (missing data)": int((dev_mask & not_saturday & ~usable).sum()),
        "signal days": int((dev_mask & signal).sum()),
        "trades": len(dev_trades),
    })
frequency = pd.DataFrame(rows).set_index("setup")
print(frequency.to_string())

       development days  excluded (Saturday entry)  skipped (missing data)  signal days  trades
setup                                                                                          
D1                 2326                        333                      90          103      84
D2                 2326                        333                       6          122     110


## Evaluation plan (not run)

Nothing below is computed until this pre-registration is approved. After approval, the BTC development evaluation is run first. The held-out test and the ETH/SOL replication are then each run **once**, with exactly these rules, thresholds, measures and baselines, and are never re-run after changes.

### Periods
- **Development:** trades whose exit day (t + 2) is on or before 2023-12-31.
- **Held-out test:** trades whose entry day (t + 1) is on or after 2024-01-01.
- The single trade entering on 2023-12-31 and exiting on 2024-01-01 belongs to neither period.
- `RV_OPEN_HIGH` stays at the development value above for the test period.

### Measures
Per setup and period, on the net one-day log return after 0.30% round-trip costs:
- number of trades, signal days, and days skipped for missing data;
- mean, median, share of trades with a positive net return;
- standard error of the mean (sample standard deviation / √n).

### Base rates
In the same period and at the same cost, every day t is treated as a one-day trade (buy at the open of t + 1, sell at the open of t + 2), with Saturday entries excluded as in the setups. Each base rate is compared with the setup mean as (setup mean − base-rate mean) / setup standard error.
1. **All days.** The mean over every eligible day.
2. **Same entry weekday.** The mean for each entry weekday, combined with weights equal to the setup's share of trades on each weekday.
3. **Same entry weekday and same previous-day trend regime.** The mean in each (entry weekday, `trend` on day t) cell, combined with weights equal to the setup's share of trades in each cell.
   - "Previous-day" means the day before entry, i.e. the signal day t.
   - Days without a `trend` label are excluded from this base rate.
   - Its weights come from the setup's trades that have a label. Every D1 trade has one; D2 trades before the label exists are counted and reported.

### Random-entry baseline matched by weekday
- For each setup and period, draw without replacement, from the eligible base-rate days of each entry weekday, as many days as the setup traded on that weekday, and record the mean net return.
- Repeat 2,000 times, with seed 0 for development and seed 1 for test (one generator per period, used for D1 then D2).
- Report the median of the random means and the share of random means greater than or equal to the setup's mean.

### ETH/SOL replication (unchanged rules)
- After the BTC held-out run, D1 and D2 are applied unchanged to ETH/USDT and SOL/USDT: the same conditions, the same Saturday exclusion, periods, costs, measures, base rates and random baseline.
- **15-minute data needed:** the US-open measures need 15-minute ETH and SOL data, not yet downloaded. It is downloaded once with `refresh_symbol(..., timeframe="15m")` before the replication, with no other change.
- **Threshold per asset:** `rv_open` is not scale-free (ETH and SOL are more volatile than BTC). The rule "above the 2/3 quantile of development US-open activity" is therefore applied with each asset's own development-period quantile, computed from its own 2017–2023 data (SOL from its first available data) before any outcome. All other conditions are sign conditions or regime labels and carry over as they are.

### Number of comparisons, stated explicitly
- **BTC:** 2 setups × (3 base rates + 1 random baseline) × 2 periods = **16** (8 development, 8 held-out).
- **ETH/SOL replication:** 2 assets × 2 setups × 4 × 2 periods = **32**.
- **Total: 48 comparisons.**
- As a reference only (not a pass/fail rule): for the 8 BTC held-out comparisons, a Bonferroni-adjusted two-sided 5% level corresponds to |difference / SE| > 2.73.

### Limitations known in advance
- **The held-out test is not fully untouched.** 2024 onward daily BTC data was already examined in Phases 4–6, and its volatility in notebooks 10–14. Its volatility figures were consistent with the Saturday exclusion, which is justified by development figures alone. Replication on ETH and SOL is the main independent check.
- **The US-open window was chosen in the intraday track** using volatility statistics that included 2024 onward (Gates 1–2).
- **Trade counts may be small:** about a third of days pass the activity filter, before the direction and trend conditions. Standard errors are reported for that reason.
- **`find_trades` with a one-day hold skips a signal on the day after a traded signal.** The trade counts above reflect that.
- **Costs:** the 0.15% per side assumption should be checked against the current exchange fee schedule.

## Updated frequency counts after the rule change (counts only, no returns)

Rule change approved before any outcome: every signal day is its own trade, built with `find_one_day_trades` (`cryptosense.backtest`) instead of `find_trades(signal, 1)`. The development counts above are recomputed with it below. No price after a signal is used.

In [3]:
from cryptosense.backtest import find_one_day_trades

rows = []
for name, setup in SETUPS_D.items():
    usable = daily[setup["inputs"]].notna().all(axis=1)
    signal = usable & setup["rule"](daily).fillna(False).astype(bool) & not_saturday
    trade_signal_days = daily.index[[entry - 1 for entry, _ in find_one_day_trades(signal)]]
    dev_mask = daily["exit_day"] <= DEV_LAST_EXIT
    rows.append({
        "setup": name,
        "signal days": int((dev_mask & signal).sum()),
        "trades (find_trades, superseded)": int(frequency.loc[name, "trades"]),
        "trades (one per signal day)": int((daily.loc[trade_signal_days, "exit_day"] <= DEV_LAST_EXIT).sum()),
    })
updated_frequency = pd.DataFrame(rows).set_index("setup")
print(updated_frequency.to_string())

       signal days  trades (find_trades, superseded)  trades (one per signal day)
setup                                                                            
D1             103                                84                          103
D2             122                               110                          122


## Decision (pre-registration)

**Decision:** D1 and D2 are approved as written, with one rule change made before any outcome was computed.

**Rule change: every signal day is its own trade.**
- A signal at the close of day t buys at the open of t + 1 and sells at the open of t + 2, including on back-to-back signal days. Each trade pays the full 0.30% round trip.
- **Reason:** one-day trades on consecutive days do not overlap. A trade exits at the same open where the next one enters, so ignoring the second signal, as `find_trades(signal, 1)` does, would drop trades for no reason.
- Trades are built with the new tested function `find_one_day_trades` in `cryptosense.backtest`. `find_trades` is unchanged, because the Phase 6 backtests depend on it.
- This replaces every reference to `find_trades(signal, 1)` in the common rules and evaluation plan above. Everything else in the evaluation plan is unchanged.

**ETH/SOL threshold rule.** For the replication, ETH/USDT and SOL/USDT use their own development-period 2/3 quantile of `rv_open`, computed from their own data up to 2023-12-31 before any outcome, as proposed in the evaluation plan.

**Updated development frequency counts** (counts only, no returns; trades exiting on or before 2023-12-31):

| Setup | Signal days | Trades with `find_trades` (superseded) | Trades, one per signal day |
|---|---|---|---|
| D1 | 103 | 84 | 103 |
| D2 | 122 | 110 | 122 |

Nothing has been evaluated. The BTC development evaluation is the next step, after a separate instruction.

## Decision (assets)

**Decision:** BTC/USDT remains the primary asset, as pre-registered. SOL/USDT is evaluated as a secondary asset with unchanged rules and its own development threshold (rv_open 2/3 quantile 0.021055; development trades: D1 69, D2 64). A pooled BTC + SOL test is added: each asset uses its own threshold and its own base rates; on days when both assets trade, their net returns are averaged into one observation, because SOL largely moves with BTC and same-day trades are not independent. SOL's preparation (notebook 16) showed its most active hour is 10:00 New York in both periods, matching the US-open window. Known limitation: with about 65 development trades and a typical daily move of roughly 6–7%, SOL alone can only detect very large edges. No one-day outcome for any asset has been computed at the time of this decision.

## Development evaluation

The evaluation plan above, as amended by the two decisions, run for the **development period only**: trades exiting on or before 2023-12-31. BTC/USDT is the primary asset and SOL/USDT the secondary asset; a pooled BTC + SOL test follows.

- **Rules:** come from `cryptosense.one_day` (`ONE_DAY_SETUPS`, `RV_OPEN_THRESHOLDS`, `us_open_measures`, `one_day_signals`), which contains D1 and D2 as approved, with BTC 0.010688 and SOL 0.021055.
- **Trades:** every signal day is its own trade (`find_one_day_trades`).
- **Nothing from 2024 onward is computed:** daily data is cut at 2023-12-31 and 15-minute data at 2024-01-01 00:00 UTC before any signal or return is calculated. With the daily data ending on 2023-12-31, every trade exits on or before that day.

**Definitions (from the pre-registration)**
- **Net log return of a trade:** log(open of t + 2 × (1 − 0.0015) / (open of t + 1 × (1 + 0.0015))).
- **Base-rate days:** every day t with a complete trade in the period whose entry day is not a Saturday, at the same cost.
- **Base rates:**
  1. All days.
  2. Same entry weekday: the mean over the setup's trades of their weekday's base mean, i.e. weighted by the setup's weekday mix.
  3. Same entry weekday and same trend on day t: the same, by (weekday, trend) cell. Trades and days without a trend label are excluded from this base rate, and the number of unlabelled trades is reported.
- **Difference in SE** = (setup mean − base mean) / setup standard error.
- **Random baseline:** for each setup, weekday-matched draws without replacement from the base-rate days, 2,000 times. One generator seeded 0 per asset is used for D1, then D2: the pre-registered procedure, applied unchanged to each asset.
- **Pooled BTC + SOL** (per setup): one observation per signal day. If both assets trade that day, their net returns are averaged.
  - Each asset uses its own threshold and base rates.
  - Each observation's base value is the average of the trading assets' per-trade base values, for each of the three base rates.
  - The pooled base rate is the mean of those values over the pooled observations (base 3 over observations with a label).

In [4]:
from cryptosense.one_day import ONE_DAY_SETUPS, RV_OPEN_THRESHOLDS, one_day_signals, us_open_measures

COST_PER_SIDE = 0.0015
N_RUNS = 2000
DEV_DAILY_END = "2023-12-31"
DEV_15M_END = pd.Timestamp("2024-01-01", tz="UTC")
ASSETS = ["BTC/USDT", "SOL/USDT"]
SETUP_NAMES = list(ONE_DAY_SETUPS)
PRE_REGISTERED_TRADES = {("BTC/USDT", "D1"): 103, ("BTC/USDT", "D2"): 122,
                         ("SOL/USDT", "D1"): 69, ("SOL/USDT", "D2"): 64}


def net_log_return(entry_open, exit_open):
    return np.log(exit_open * (1 - COST_PER_SIDE) / (entry_open * (1 + COST_PER_SIDE)))


asset_base, asset_trades = {}, {}
count_rows = []
for symbol in ASSETS:
    dev_daily = add_regimes(add_features(load_processed(symbol, DATA_DIR))).loc[:DEV_DAILY_END]
    dev_m15 = load_processed(symbol, DATA_DIR, timeframe="15m")
    dev_m15 = dev_m15[dev_m15.index < DEV_15M_END]
    signals = one_day_signals(dev_daily, us_open_measures(dev_m15), symbol)

    opens = dev_daily["open"].to_numpy()
    n = len(dev_daily)
    t = np.arange(n - 2)
    base = pd.DataFrame({
        "signal_day": dev_daily.index[t],
        "entry_weekday": (dev_daily.index[t] + pd.Timedelta(days=1)).day_name(),
        "trend": dev_daily["trend"].to_numpy()[t],
        "net": net_log_return(opens[t + 1], opens[t + 2]),
    })
    base = base[base["entry_weekday"] != "Saturday"].reset_index(drop=True)
    asset_base[symbol] = base

    for name in SETUP_NAMES:
        pairs = find_one_day_trades(signals[name])
        trades = pd.DataFrame({
            "signal_day": dev_daily.index[[e - 1 for e, _ in pairs]],
            "entry_weekday": dev_daily.index[[e for e, _ in pairs]].day_name(),
            "trend": dev_daily["trend"].to_numpy()[[e - 1 for e, _ in pairs]],
            "net": [net_log_return(opens[e], opens[x]) for e, x in pairs],
            "exit_day": dev_daily.index[[x for _, x in pairs]],
        })
        asset_trades[(symbol, name)] = trades
        count_rows.append({"asset": symbol, "setup": name, "trades": len(trades),
                           "pre-registered": PRE_REGISTERED_TRADES[(symbol, name)],
                           "last exit": trades["exit_day"].max().date()})

counts = pd.DataFrame(count_rows).set_index(["asset", "setup"])
print(counts.to_string())
mismatch = counts[counts["trades"] != counts["pre-registered"]]
if len(mismatch):
    raise RuntimeError(f"Trade counts differ from the pre-registration; stopping:\n{mismatch}")
print("\nTrade counts match the pre-registration.")

                trades  pre-registered   last exit
asset    setup                                    
BTC/USDT D1        103             103  2023-12-22
         D2        122             122  2023-11-11
SOL/USDT D1         69              69  2023-12-27
         D2         64              64  2023-12-01

Trade counts match the pre-registration.


### Setup trades: net one-day log return after costs (development)

Mean, median and standard error in percent (log return × 100).

In [5]:
def measures(r):
    """n, mean, median, share positive and standard error of net log returns (in %)."""
    r = pd.Series(r, dtype=float).dropna()
    return {"n": len(r), "mean_%": 100 * r.mean(), "median_%": 100 * r.median(),
            "share_positive": (r > 0).mean(), "se_%": 100 * r.std() / np.sqrt(len(r))}


trade_table = pd.DataFrame([
    {"asset": symbol, "setup": name, **measures(asset_trades[(symbol, name)]["net"])}
    for symbol in ASSETS for name in SETUP_NAMES
]).set_index(["asset", "setup"])
print(trade_table.to_string(formatters={"n": "{:,}".format, "share_positive": "{:.3f}".format,
                                        **{c: "{:.4f}".format for c in ["mean_%", "median_%", "se_%"]}}))

                 n  mean_% median_% share_positive   se_%
asset    setup                                           
BTC/USDT D1    103 -0.1353  -0.3708          0.447 0.4255
         D2    122 -1.0233  -1.2327          0.393 0.4892
SOL/USDT D1     69  0.4600  -0.2685          0.464 1.2929
         D2     64  1.3786   1.1891          0.562 1.1058


### Base rates and differences in standard errors (development)

In [6]:
def add_base_values(trades, base):
    """Per-trade base values: all days, same weekday, same weekday and trend (NaN without a label)."""
    by_weekday = base.groupby("entry_weekday")["net"].mean()
    by_cell = base.dropna(subset=["trend"]).groupby(["entry_weekday", "trend"])["net"].mean()
    out = trades.copy()
    out["base_1"] = base["net"].mean()
    out["base_2"] = out["entry_weekday"].map(by_weekday)
    out["base_3"] = [by_cell.get((w, tr), np.nan) if isinstance(tr, str) else np.nan
                     for w, tr in zip(out["entry_weekday"], out["trend"])]
    return out


BASE_LABELS = {"base_1": "1. all days", "base_2": "2. same entry weekday",
               "base_3": "3. same weekday and day-t trend"}
rows, base_cells = [], []
for symbol in ASSETS:
    base = asset_base[symbol]
    base_cells.append({"asset": symbol, "base-rate days": len(base), "all-days mean_%": 100 * base["net"].mean(),
                       "days with trend label": int(base["trend"].notna().sum())})
    for name in SETUP_NAMES:
        tr = add_base_values(asset_trades[(symbol, name)], base)
        asset_trades[(symbol, name)] = tr
        m = measures(tr["net"])
        mean, se = m["mean_%"] / 100, m["se_%"] / 100
        for col, label in BASE_LABELS.items():
            b = tr[col].mean()   # mean over trades = weighted by the setup's mix (base 3: labelled trades)
            rows.append({"asset": symbol, "setup": name, "base rate": label,
                         "setup_mean_%": 100 * mean, "setup_se_%": 100 * se, "base_mean_%": 100 * b,
                         "diff_%": 100 * (mean - b), "diff_in_se": (mean - b) / se,
                         "trades used for base": int(tr[col].notna().sum())})

print(pd.DataFrame(base_cells).set_index("asset").to_string(float_format="{:.4f}".format))
print()
base_table = pd.DataFrame(rows).set_index(["asset", "setup", "base rate"])
print(base_table.to_string(float_format="{:.4f}".format))

          base-rate days  all-days mean_%  days with trend label
asset                                                           
BTC/USDT            1993          -0.2137                   1908
SOL/USDT            1059          -0.0533                    974

                                                setup_mean_%  setup_se_%  base_mean_%  diff_%  diff_in_se  trades used for base
asset    setup base rate                                                                                                       
BTC/USDT D1    1. all days                           -0.1353      0.4255      -0.2137  0.0784      0.1843                   103
               2. same entry weekday                 -0.1353      0.4255      -0.2190  0.0837      0.1968                   103
               3. same weekday and day-t trend       -0.1353      0.4255       0.0629 -0.1982     -0.4658                   103
         D2    1. all days                           -1.0233      0.4892      -0.2137 -0.8096     -

### Weekday-matched random-entry baseline (development, 2,000 draws, seed 0 per asset)

In [7]:
rows = []
for symbol in ASSETS:
    base = asset_base[symbol]
    pools = {w: g["net"].to_numpy() for w, g in base.groupby("entry_weekday")}
    rng = np.random.default_rng(0)
    for name in SETUP_NAMES:
        tr = asset_trades[(symbol, name)]
        k = tr["entry_weekday"].value_counts()
        actual = tr["net"].mean()
        sims = np.array([
            np.concatenate([rng.choice(pools[w], size=int(c), replace=False) for w, c in k.items()]).mean()
            for _ in range(N_RUNS)
        ])
        rows.append({"asset": symbol, "setup": name, "trades": len(tr), "setup_mean_%": 100 * actual,
                     "random_median_%": 100 * np.median(sims),
                     "share_random_ge_setup": (sims >= actual).mean()})
random_table = pd.DataFrame(rows).set_index(["asset", "setup"])
print(random_table.to_string(float_format="{:.4f}".format))

                trades  setup_mean_%  random_median_%  share_random_ge_setup
asset    setup                                                              
BTC/USDT D1        103       -0.1353          -0.2196                 0.4265
         D2        122       -1.0233          -0.2044                 0.9860
SOL/USDT D1         69        0.4600          -0.1143                 0.2485
         D2         64        1.3786          -0.1000                 0.0520


### Pooled BTC + SOL (development)

One observation per signal day; if both assets trade that day, their net returns (and their per-trade base values) are averaged.

In [8]:
rows = []
for name in SETUP_NAMES:
    stacked = pd.concat([asset_trades[(s, name)].assign(asset=s) for s in ASSETS])
    pooled = stacked.groupby("signal_day").agg(
        net=("net", "mean"), assets=("asset", "nunique"),
        base_1=("base_1", "mean"), base_2=("base_2", "mean"), base_3=("base_3", "mean"),
    )
    m = measures(pooled["net"])
    mean, se = m["mean_%"] / 100, m["se_%"] / 100
    for col, label in BASE_LABELS.items():
        b = pooled[col].mean()
        rows.append({"setup": name, "base rate": label, "observations": len(pooled),
                     "both-asset days": int((pooled["assets"] == 2).sum()),
                     "pooled_mean_%": 100 * mean, "pooled_se_%": 100 * se, "base_mean_%": 100 * b,
                     "diff_%": 100 * (mean - b), "diff_in_se": (mean - b) / se,
                     "observations used for base": int(pooled[col].notna().sum())})
pooled_table = pd.DataFrame(rows).set_index(["setup", "base rate"])
print(pooled_table.to_string(float_format="{:.4f}".format))

                                       observations  both-asset days  pooled_mean_%  pooled_se_%  base_mean_%  diff_%  diff_in_se  observations used for base
setup base rate                                                                                                                                              
D1    1. all days                               153               19        -0.0617       0.5874      -0.1514  0.0896      0.1526                         153
      2. same entry weekday                     153               19        -0.0617       0.5874      -0.1769  0.1152      0.1961                         153
      3. same weekday and day-t trend           153               19        -0.0617       0.5874       0.1071 -0.1688     -0.2874                         153
D2    1. all days                               166               20        -0.2416       0.5077      -0.1616 -0.0801     -0.1578                         166
      2. same entry weekday                     166 

## Development results

Development period only: trades exiting on or before 2023-12-31. Net one-day log returns after 0.15% costs per side, in %. Difference in SE = (setup mean − base mean) / setup SE. Trade counts match the pre-registration (BTC D1 103, D2 122; SOL D1 69, D2 64).

### Setup trades

| Asset | Setup | Trades | Mean | Median | Share positive | SE |
|---|---|---|---|---|---|---|
| BTC/USDT | D1 | 103 | −0.1353 | −0.3708 | 0.447 | 0.4255 |
| BTC/USDT | D2 | 122 | −1.0233 | −1.2327 | 0.393 | 0.4892 |
| SOL/USDT | D1 | 69 | 0.4600 | −0.2685 | 0.464 | 1.2929 |
| SOL/USDT | D2 | 64 | 1.3786 | 1.1891 | 0.562 | 1.1058 |

Base-rate days (non-Saturday entries): BTC 1,993 (all-days mean −0.2137; 1,908 with a trend label); SOL 1,059 (all-days mean −0.0533; 974 with a trend label).

### Comparisons with base rates

| Asset | Setup | Base rate | Base mean | Difference | Difference in SE | Trades used |
|---|---|---|---|---|---|---|
| BTC/USDT | D1 | 1. all days | −0.2137 | 0.0784 | 0.18 | 103 |
| BTC/USDT | D1 | 2. same entry weekday | −0.2190 | 0.0837 | 0.20 | 103 |
| BTC/USDT | D1 | 3. same weekday and day-t trend | 0.0629 | −0.1982 | −0.47 | 103 |
| BTC/USDT | D2 | 1. all days | −0.2137 | −0.8096 | −1.65 | 122 |
| BTC/USDT | D2 | 2. same entry weekday | −0.2077 | −0.8156 | −1.67 | 122 |
| BTC/USDT | D2 | 3. same weekday and day-t trend | −0.2888 | −0.7345 | −1.50 | 112 |
| SOL/USDT | D1 | 1. all days | −0.0533 | 0.5133 | 0.40 | 69 |
| SOL/USDT | D1 | 2. same entry weekday | −0.1157 | 0.5757 | 0.45 | 69 |
| SOL/USDT | D1 | 3. same weekday and day-t trend | 0.1979 | 0.2621 | 0.20 | 69 |
| SOL/USDT | D2 | 1. all days | −0.0533 | 1.4319 | 1.29 | 64 |
| SOL/USDT | D2 | 2. same entry weekday | −0.0795 | 1.4580 | 1.32 | 64 |
| SOL/USDT | D2 | 3. same weekday and day-t trend | 0.0555 | 1.3231 | 1.20 | 58 |

### Weekday-matched random-entry baseline (2,000 draws, seed 0 per asset)

| Asset | Setup | Trades | Setup mean | Random median | Share of random means ≥ setup mean |
|---|---|---|---|---|---|
| BTC/USDT | D1 | 103 | −0.1353 | −0.2196 | 0.4265 |
| BTC/USDT | D2 | 122 | −1.0233 | −0.2044 | 0.9860 |
| SOL/USDT | D1 | 69 | 0.4600 | −0.1143 | 0.2485 |
| SOL/USDT | D2 | 64 | 1.3786 | −0.1000 | 0.0520 |

### Pooled BTC + SOL

| Setup | Observations (both-asset days) | Pooled mean | Pooled SE | Base rate | Base mean | Difference in SE |
|---|---|---|---|---|---|---|
| D1 | 153 (19) | −0.0617 | 0.5874 | 1. all days | −0.1514 | 0.15 |
| D1 | 153 (19) | −0.0617 | 0.5874 | 2. same entry weekday | −0.1769 | 0.20 |
| D1 | 153 (19) | −0.0617 | 0.5874 | 3. same weekday and day-t trend | 0.1071 | −0.29 |
| D2 | 166 (20) | −0.2416 | 0.5077 | 1. all days | −0.1616 | −0.16 |
| D2 | 166 (20) | −0.2416 | 0.5077 | 2. same entry weekday | −0.1616 | −0.16 |
| D2 | 166 (20) | −0.2416 | 0.5077 | 3. same weekday and day-t trend | −0.1527 | −0.18 |

Pooled base 3 uses observations with a trend label: D1 153 of 153, D2 150 of 166.

## Decision (development gate)

**Decision:** The primary pre-registered test (BTC) showed no positive edge: D1 was indistinguishable from base rates and random entries, and D2 was worse than 98.6% of weekday-matched random draws. D1 is closed for all assets. D2 is closed for BTC.

**Exploratory hypothesis (registered after seeing development results):** On SOL/USDT, D2 — exactly as pre-registered (rv_open above 0.021055, ret_open below 0, day t closed up, Saturday entries excluded, one-day hold, 0.30% round-trip costs) — beats base rates on held-out data (trades entering from 2024-01-01). It was selected because it looked best among 24 development comparisons, and the same rule had the opposite sign on BTC; it is therefore exploratory, not confirmatory.

**Success criterion, fixed now:** the held-out difference from base rate 3 (same weekday and day-t trend) is at least +2 standard errors, AND at most 5% of weekday-matched random draws (2,000, seed 1) are at or above the setup mean. Anything else is recorded as not replicated. The test is run once, with no changes to the rule, threshold, costs or method.